In [1]:
"""
Q1. Embedding a query

Embed the following query:

    How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

    -0.31
    -0.02
    0.12
    0.44
"""

q = "How does approximate nearest neighbor search work?"

from embedder import Embedder

embed = Embedder()

v = embed.encode(q)
v[0]


2026-07-18 14:50:31.097631485 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


np.float64(-0.02058203437252893)

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [3]:
"""
Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

    0.07
    0.37
    0.68
    0.92

"""
dv = []
for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        dv = embed.encode(doc["content"])
        
        
if (dv.any()):
    print(v.dot(dv))

0.36107027225589694


In [4]:
len(documents)

72

In [8]:
"""
Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

scores = X.dot(v)

Which file does the highest-scoring chunk belong to (its filename)?

    02-vector-search/lessons/03-embeddings-dataset.md
    02-vector-search/lessons/06-rag-vector.md
    02-vector-search/lessons/07-sqlitesearch-vector.md
    02-vector-search/lessons/09-onnx-embedder.md
"""
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

texts = [chunk["filename"] + " " + chunk["content"] for chunk in chunks]

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
scores = X.dot(v)
idx = np.argmax(scores)

chunks[idx]

  0%|          | 0/6 [00:00<?, ?it/s]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [12]:
"""
Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

    What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

    02-vector-search/lessons/04-vector-search.md
    04-evaluation/lessons/05-search-metrics.md
    04-evaluation/lessons/13-llm-as-judge.md
    05-monitoring/lessons/04-metrics.md

"""
from minsearch import VectorSearch

q2 = "What metric do we use to evaluate a search engine?"

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

q2_vector = embed.encode(q2)

results = vindex.search(q2_vector, num_results=5)
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [16]:
"""
Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

    How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

    02-vector-search/lessons/01-intro.md
    02-vector-search/lessons/02-embeddings.md
    02-vector-search/lessons/08-pgvector.md
    03-orchestration/lessons/05-rag.md

"""
new_query = "How do I store vectors in PostgreSQL?"

from minsearch import Index

index = Index(keyword_fields=["filename"], text_fields=["content"])
index.fit(chunks)

non_vector_results = index.search(new_query, num_results=5)

new_query_v = embed.encode(new_query)

vector_results = vindex.search(new_query_v, num_results=5)


In [17]:
non_vector_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [18]:
vector_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [ ]:
"""
Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Now run the query "How do I give the model access to tools?" with vector and text search and fuse the results with rrf:

results = rrf([vector_results, text_results])

Which file is ranked first after RRF?

    01-agentic-rag/lessons/01-intro.md
    01-agentic-rag/lessons/13-function-calling.md
    01-agentic-rag/lessons/14-agentic-loop.md
    01-agentic-rag/lessons/16-other-frameworks.md

Notice that this file isn't first in either search on its own - it wins because it ranks high in both.
"""

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

rrf_q = "How do I give the model access to tools?"
rrf_q_v = embed.encode(rrf_q)

text_results = index.search(rrf_q, num_results=5)
vector_results = vindex.search(rrf_q_v, num_results=5)

results = rrf([vector_results, text_results])
results

AttributeError: 'str' object has no attribute 'reshape'